# Preparation des donnees — Modele CLV Nouvelles Acquisitions

## Sources de donnees (`1. DATA/`)

| Dossier | Fichiers | Contenu |
|---------|----------|---------|
| `STOCK/{YYYY}/` | `STOCK_YY_MM.csv` (1 par mois) | Clients acquis ce mois-la. Annee/mois deduits du nom de fichier. |
| `PNB/` | `PNB_YYYY_SX.csv` (X=1,2,3) | NBI mensuel de **tous** les clients. 4 colonnes-mois par fichier (indices bancaires). |
| `equipements/` | `YYYY QX.csv` (1 par trimestre) | Snapshot equipements a la fin de chaque trimestre. |

## Encodage des mois bancaires
Janvier 1994 = indice 1. Formule : `indice = (annee - 1994) * 12 + mois`. Dec 2025 = 384.

## Sortie
DataFrame `df` + fichier `df_acquisitions.pkl`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import re
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

DATA_DIR = Path("1. DATA")
STOCK_DIR = DATA_DIR / "STOCK"
PNB_DIR = DATA_DIR / "PNB"
EQUIP_DIR = DATA_DIR / "equipements"

EPOCH_YEAR = 1994

def date_to_idx(year, month):
    return (year - EPOCH_YEAR) * 12 + month

def idx_to_yearmonth(idx):
    year = EPOCH_YEAR + (idx - 1) // 12
    month = (idx - 1) % 12 + 1
    return year, month

assert date_to_idx(2023, 1) == 349
assert date_to_idx(2025, 12) == 384
assert idx_to_yearmonth(384) == (2025, 12)
print("Encodage bancaire OK : Jan 2023 =", date_to_idx(2023, 1), "| Dec 2025 =", date_to_idx(2025, 12))

In [ ]:
def load_csv(path):
    path = Path(path)
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            return pd.read_csv(path, encoding=enc, sep=None, engine="python")
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Impossible de lire : {path}")


def detect_client_id(df):
    candidates = ["client_id", "clientid", "id_client", "customer_id",
                  "cust_id", "no_client", "num_client", "id"]
    for col in df.columns:
        if col.lower().strip() in candidates:
            return col
    return df.columns[0]


def standardize_id(df, id_col):
    df = df.rename(columns={id_col: "client_id"})
    df["client_id"] = df["client_id"].astype(str).str.strip()
    df = df[df["client_id"].notna() & (df["client_id"] != "") & (df["client_id"] != "nan")]
    return df


def detect_month_idx_columns(df):
    idx_cols = []
    for col in df.columns:
        try:
            val = int(col)
            if 200 <= val <= 600:
                idx_cols.append(val)
        except (ValueError, TypeError):
            pass
    return sorted(idx_cols)


print("Helpers OK")

In [ ]:
# ── STOCK : un fichier par mois d'acquisition ────────────────────────────────
print("=" * 60)
print("CHARGEMENT DES FICHIERS STOCK")
print("=" * 60)

stock_frames = []
pattern = re.compile(r"STOCK_(\d{2})_(\d{2})", re.IGNORECASE)

for year_folder in sorted(STOCK_DIR.iterdir()):
    if not year_folder.is_dir():
        continue
    for csv_file in sorted(year_folder.glob("*.csv")):
        m = pattern.search(csv_file.stem)
        if not m:
            print(f"  Skip (nom non reconnu) : {csv_file.name}")
            continue

        yy, mm = int(m.group(1)), int(m.group(2))
        full_year = 2000 + yy
        period = date_to_idx(full_year, mm)

        raw = load_csv(csv_file)
        id_col = detect_client_id(raw)
        raw = standardize_id(raw, id_col)

        raw["annee_acquisition"] = full_year
        raw["mois_acquisition"] = mm
        raw["trimestre_acquisition"] = f"Q{(mm - 1) // 3 + 1}"
        raw["period"] = period

        stock_frames.append(raw)
        print(f"  {csv_file.name} -> {full_year}-{mm:02d} (period={period}) | {len(raw):,} clients")

stock_df = pd.concat(stock_frames, ignore_index=True)
stock_df = stock_df.drop_duplicates(subset="client_id", keep="first")

# Standardiser segment / profession
for variants, std in [(["segment", "seg", "categorie_client", "classe"], "segment"),
                       (["profession", "csp", "categorie_socio", "job", "metier"], "profession")]:
    for v in variants:
        matches = [c for c in stock_df.columns if c.lower().strip() == v and c != std]
        if matches:
            stock_df = stock_df.rename(columns={matches[0]: std})
            break

print(f"\nTotal nouveaux clients STOCK : {len(stock_df):,}")
print(f"Colonnes : {list(stock_df.columns)}")
print(f"\nPar annee :")
print(stock_df["annee_acquisition"].value_counts().sort_index().to_string())
print(f"\nPar trimestre :")
print(stock_df.groupby(["annee_acquisition", "trimestre_acquisition"])["client_id"].count().to_string())
if "segment" in stock_df.columns:
    print(f"\nSegments :")
    print(stock_df["segment"].value_counts().to_string())

In [ ]:
# ── PNB : fusionner tous les fichiers en une matrice client x mois ────────────
print("=" * 60)
print("CHARGEMENT DES FICHIERS PNB")
print("=" * 60)

pnb_frames = []
all_month_idx = set()
pnb_pattern = re.compile(r"PNB_(\d{4})_S(\d)", re.IGNORECASE)

for csv_file in sorted(PNB_DIR.glob("*.csv")):
    m = pnb_pattern.search(csv_file.stem)
    if not m:
        print(f"  Skip (nom non reconnu) : {csv_file.name}")
        continue

    yyyy, sx = int(m.group(1)), int(m.group(2))
    raw = load_csv(csv_file)
    id_col = detect_client_id(raw)
    raw = standardize_id(raw, id_col)

    month_cols = detect_month_idx_columns(raw)
    if not month_cols:
        print(f"  ERREUR : aucun indice mois dans {csv_file.name}")
        print(f"  Colonnes : {list(raw.columns[:15])}")
        continue

    all_month_idx.update(month_cols)
    keep = ["client_id"] + [str(c) for c in month_cols]
    pnb_frames.append(raw[keep])

    rng = [idx_to_yearmonth(min(month_cols)), idx_to_yearmonth(max(month_cols))]
    print(f"  {csv_file.name} -> {rng[0][0]}-{rng[0][1]:02d} a {rng[1][0]}-{rng[1][1]:02d} "
          f"({len(month_cols)} mois, {len(raw):,} clients)")

if not pnb_frames:
    raise FileNotFoundError(f"Aucun fichier PNB trouve dans {PNB_DIR}")

# Fusion progressive sur client_id (outer pour garder tous les clients et tous les mois)
print("\nFusion des fichiers PNB...")
pnb_wide = pnb_frames[0]
for frame in pnb_frames[1:]:
    pnb_wide = pd.merge(pnb_wide, frame, on="client_id", how="outer")

month_str_cols = sorted([str(c) for c in all_month_idx], key=lambda x: int(x))
for col in month_str_cols:
    if col in pnb_wide.columns:
        pnb_wide[col] = pd.to_numeric(pnb_wide[col], errors="coerce")

rng_min, rng_max = idx_to_yearmonth(min(all_month_idx)), idx_to_yearmonth(max(all_month_idx))
print(f"\nMatrice PNB : {len(pnb_wide):,} clients x {len(month_str_cols)} mois")
print(f"Plage : {rng_min[0]}-{rng_min[1]:02d} (idx {min(all_month_idx)}) "
      f"a {rng_max[0]}-{rng_max[1]:02d} (idx {max(all_month_idx)})")

In [ ]:
# ── Jointure STOCK + PNB : extraire 12 mois de NBI par client ────────────────
print("=" * 60)
print("EXTRACTION DES 12 MOIS DE PNB PAR CLIENT")
print("=" * 60)

df_work = pd.merge(stock_df, pnb_wide, on="client_id", how="left")
n_matched = df_work[month_str_cols[0]].notna().sum() if month_str_cols else 0
print(f"Jointure STOCK x PNB : {len(df_work):,} clients ({n_matched:,} trouves dans PNB)")

# Pour chaque client : pnb_m1 = mois d'acquisition, pnb_m2 = mois suivant, ..., pnb_m12
print("Construction pnb_m1..pnb_m12 ...")

for m in range(1, 13):
    col_name = f"pnb_m{m}"
    df_work[col_name] = df_work.apply(
        lambda r: r.get(str(int(r["period"]) + m - 1), np.nan)
                  if pd.notna(r.get("period")) else np.nan,
        axis=1
    )

# Supprimer les colonnes indices brutes
df_work = df_work.drop(columns=[c for c in month_str_cols if c in df_work.columns])

# Cumuls
df_work["pnb_cumul_3m"]  = df_work[[f"pnb_m{i}" for i in range(1, 4)]].sum(axis=1, min_count=1)
df_work["pnb_cumul_6m"]  = df_work[[f"pnb_m{i}" for i in range(1, 7)]].sum(axis=1, min_count=1)
df_work["pnb_12m_total"] = df_work[[f"pnb_m{i}" for i in range(1, 13)]].sum(axis=1, min_count=1)

# Couverture
for n, label in [(3, "3m"), (6, "6m"), (12, "12m")]:
    has = df_work[[f"pnb_m{i}" for i in range(1, n+1)]].notna().all(axis=1)
    print(f"  {label} complets : {has.sum():,} / {len(df_work):,} ({has.mean()*100:.1f}%)")

print(f"\nPNB 12m — moy: {df_work['pnb_12m_total'].mean():,.0f} EUR | "
      f"mediane: {df_work['pnb_12m_total'].median():,.0f} EUR")

In [ ]:
# ── EQUIPEMENTS : snapshots trimestriels ─────────────────────────────────────
print("=" * 60)
print("CHARGEMENT EQUIPEMENTS (snapshots trimestriels)")
print("=" * 60)

equip_feat = None

if EQUIP_DIR.exists():
    equip_snapshots = {}
    snap_pattern = re.compile(r"(\d{4})\s*Q(\d)", re.IGNORECASE)

    for csv_file in sorted(EQUIP_DIR.glob("*.csv")):
        m = snap_pattern.search(csv_file.stem)
        if not m:
            print(f"  Skip (nom non reconnu) : {csv_file.name}")
            continue
        yyyy, qtr = int(m.group(1)), int(m.group(2))
        key = (yyyy, qtr)
        raw = load_csv(csv_file)
        id_col = detect_client_id(raw)
        raw = standardize_id(raw, id_col)
        equip_snapshots[key] = raw
        print(f"  {csv_file.name} -> {yyyy} Q{qtr} | {len(raw):,} clients, {len(raw.columns)} cols")

    print(f"\n  {len(equip_snapshots)} snapshots charges")

    if equip_snapshots:
        # For each client, pick the snapshot at the end of their 6-month training window.
        # Acquisition in month M -> training window ends at M+5 -> use the quarter that contains M+5.
        # Quarter mapping: Q1=Jan-Mar, Q2=Apr-Jun, Q3=Jul-Sep, Q4=Oct-Dec

        def training_end_quarter(row):
            acq_year = int(row["annee_acquisition"])
            acq_month = int(row["mois_acquisition"])
            end_month = acq_month + 5
            end_year = acq_year + (end_month - 1) // 12
            end_month = ((end_month - 1) % 12) + 1
            end_qtr = (end_month - 1) // 3 + 1
            return (end_year, end_qtr)

        available_keys = sorted(equip_snapshots.keys())

        def best_snapshot(target):
            if target in equip_snapshots:
                return target
            # Fallback: closest available snapshot that is <= target, else earliest after
            before = [k for k in available_keys if k <= target]
            if before:
                return before[-1]
            after = [k for k in available_keys if k > target]
            return after[0] if after else None

        PRODUCT_KW = ["compte", "carte", "epargne", "credit", "assurance",
                      "livret", "prevoyance", "pret", "hypothe", "bourse",
                      "vie", "deces", "invalidite", "retraite"]

        def build_equip_features(snap_df):
            product_cols = []
            for col in snap_df.columns:
                cl = col.lower()
                if cl == "client_id":
                    continue
                if any(kw in cl for kw in PRODUCT_KW):
                    product_cols.append(col)

            feat = snap_df[["client_id"]].copy()
            has_cols = []
            for col in product_cols:
                base = re.sub(r"[^a-z0-9]+", "_", col.lower()).strip("_")
                has_name = f"has_{base}"
                feat[has_name] = pd.to_numeric(snap_df[col], errors="coerce").fillna(0).gt(0).astype(int)
                has_cols.append(has_name)

            feat["n_produits_total"] = feat[has_cols].sum(axis=1) if has_cols else 0

            FAMILIES = {
                "fam_banque_quotidienne":     ["compte", "carte"],
                "fam_epargne_investissement": ["epargne", "livret", "vie", "bourse", "retraite"],
                "fam_credit":                 ["credit", "pret", "hypothe"],
                "fam_assurance_protection":   ["assurance", "prevoyance", "deces", "invalidite"],
            }
            for fam, kws in FAMILIES.items():
                fam_has = [c for c in has_cols if any(kw in c for kw in kws)]
                feat[fam] = feat[fam_has].max(axis=1) if fam_has else 0
            feat["n_familles_produits"] = feat[list(FAMILIES.keys())].sum(axis=1)

            FLOW_KW = ["entree", "sortie", "inflow", "outflow", "solde", "encours", "balance", "flux"]
            for col in snap_df.columns:
                if any(kw in col.lower() for kw in FLOW_KW) and col != "client_id":
                    feat[col] = pd.to_numeric(snap_df[col], errors="coerce")

            return feat

        # Group clients by their target snapshot
        snapshot_assignments = {}
        for idx, row in df_work.iterrows():
            target = training_end_quarter(row)
            snap_key = best_snapshot(target)
            if snap_key:
                snapshot_assignments.setdefault(snap_key, []).append(row["client_id"])

        # Build equipment features per snapshot, then concatenate
        equip_parts = []
        for snap_key in sorted(snapshot_assignments.keys()):
            client_ids = set(snapshot_assignments[snap_key])
            snap_df = equip_snapshots[snap_key]
            snap_df_filtered = snap_df[snap_df["client_id"].isin(client_ids)]
            if len(snap_df_filtered) == 0:
                continue
            feat = build_equip_features(snap_df_filtered)
            equip_parts.append(feat)
            print(f"  Snapshot {snap_key[0]} Q{snap_key[1]} -> {len(snap_df_filtered):,} clients matches")

        if equip_parts:
            equip_feat = pd.concat(equip_parts, ignore_index=True)
            equip_feat = equip_feat.drop_duplicates(subset="client_id", keep="first")
            has_cols = [c for c in equip_feat.columns if c.startswith("has_")]
            print(f"\n  Features equipement : {has_cols}")
            print(f"  Total clients avec equipement : {len(equip_feat):,}")
            print(f"  n_produits_total :\n{equip_feat['n_produits_total'].value_counts().sort_index().to_string()}")
        else:
            print("  Aucun client matche dans les snapshots.")
    else:
        print("  Aucun snapshot trouve.")
else:
    print(f"  {EQUIP_DIR} introuvable — equipements ignores.")

In [ ]:
# ── Fusion finale ────────────────────────────────────────────────────────────
print("=" * 60)
print("FUSION FINALE")
print("=" * 60)

df = df_work.copy()

if equip_feat is not None:
    df = pd.merge(df, equip_feat, on="client_id", how="left")
    n_eq = df["n_produits_total"].notna().sum()
    print(f"Avec equipement : {n_eq:,} / {len(df):,} ({n_eq/len(df)*100:.1f}%)")

n_dup = df.duplicated(subset="client_id").sum()
if n_dup > 0:
    print(f"Doublons : {n_dup} -> deduplication")
    df = df.drop_duplicates(subset="client_id", keep="first")

print(f"DataFrame final : {df.shape[0]:,} x {df.shape[1]}")

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────
print("=" * 60)
print("FEATURE ENGINEERING")
print("=" * 60)

# Trajectoire PNB 6 mois
pnb_6m = [f"pnb_m{i}" for i in range(1, 7) if f"pnb_m{i}" in df.columns]

if len(pnb_6m) >= 3:
    from sklearn.linear_model import LinearRegression

    def slope(row):
        vals = np.array([row[c] for c in pnb_6m], dtype=float)
        ok = ~np.isnan(vals)
        if ok.sum() < 2:
            return 0.0
        x = np.where(ok)[0].reshape(-1, 1)
        return LinearRegression().fit(x, vals[ok]).coef_[0]

    df["pnb_pente"] = df.apply(slope, axis=1)
    df["pnb_volatilite_mensuelle"] = df[pnb_6m].std(axis=1)
    df["mois_pnb_positif"] = (df[pnb_6m] > 0).sum(axis=1)

    m1, m3 = pnb_6m[0], pnb_6m[2]
    df["is_slow_starter"] = ((df[m1] < df[m1].median()) & (df[m3] > df[m3].median())).astype(int)
    df["is_early_peak"] = ((df[m1] > df[m1].quantile(0.75)) & (df["pnb_pente"] < -20)).astype(int)

    print(f"pnb_pente moy: {df['pnb_pente'].mean():.2f}")
    print(f"mois_pnb_positif moy: {df['mois_pnb_positif'].mean():.1f}")
    print(f"slow_starters: {df['is_slow_starter'].sum():,} ({df['is_slow_starter'].mean()*100:.1f}%)")

# Anciennete
REF = pd.Timestamp("2026-03-31")
if "annee_acquisition" in df.columns and "mois_acquisition" in df.columns:
    df["date_creation"] = pd.to_datetime(
        df["annee_acquisition"].astype(str) + "-" + df["mois_acquisition"].astype(str) + "-01",
        errors="coerce"
    )
    df["mois_anciennete"] = ((REF - df["date_creation"]).dt.days / 30.44).round(0)

print(f"\nShape : {df.shape}")

In [ ]:
# ── Eligibilite entrainement ─────────────────────────────────────────────────
print("=" * 60)
print("ELIGIBILITE ENTRAINEMENT vs SCORING")
print("=" * 60)

# Eligible : acquisition <= mars 2025 (12 mois observables jusqu'a mars 2026)
CUTOFF = pd.Timestamp("2025-03-31")

if "date_creation" in df.columns:
    df["est_eligible_training"] = df["date_creation"] <= CUTOFF
    print(f"Entrainement (acq <= mars 2025) : {df['est_eligible_training'].sum():,}")
    print(f"Scoring (acq > mars 2025)       : {(~df['est_eligible_training']).sum():,}")

    print("\nCohortes eligibles :")
    cohorts = (
        df[df["est_eligible_training"]]
        .groupby(["annee_acquisition", "trimestre_acquisition"])["client_id"]
        .count().reset_index()
    )
    cohorts.columns = ["annee", "trimestre", "n_clients"]
    print(cohorts.to_string(index=False))

In [ ]:
# ── Export ────────────────────────────────────────────────────────────────────
print("=" * 60)
print("EXPORT")
print("=" * 60)

# Nettoyage colonnes intermediaires
df = df.drop(columns=[c for c in ["mois_acquisition"] if c in df.columns])

print(f"Shape finale : {df.shape[0]:,} x {df.shape[1]}")
print("\nColonnes et completude :")
for col in sorted(df.columns):
    pct_null = df[col].isna().mean() * 100
    flag = " <<< ATTENTION" if pct_null > 30 else ""
    print(f"  {col:<42} {str(df[col].dtype):<10} null={pct_null:5.1f}%{flag}")

if "pnb_12m_total" in df.columns:
    print("\nPNB 12m (clients eligibles) :")
    subset = df.loc[df["est_eligible_training"]] if "est_eligible_training" in df.columns else df
    print(subset["pnb_12m_total"].describe().round(0).to_string())

out = Path("df_acquisitions.pkl")
with open(out, "wb") as f:
    pickle.dump(df, f)
print(f"\nExporte : {out}")
print("Usage : import pickle; df = pickle.load(open('df_acquisitions.pkl', 'rb'))")

In [ ]:
# ── Apercu graphique ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Volume par trimestre
if "annee_acquisition" in df.columns:
    vol = df.groupby(["annee_acquisition", "trimestre_acquisition"])["client_id"].count().reset_index()
    vol["label"] = vol["annee_acquisition"].astype(str) + "-" + vol["trimestre_acquisition"]
    axes[0, 0].bar(vol["label"], vol["client_id"], color="steelblue", edgecolor="white")
    axes[0, 0].set_title("Nouveaux clients par trimestre")
    axes[0, 0].set_ylabel("Clients")
    axes[0, 0].tick_params(axis="x", rotation=45)
    axes[0, 0].grid(axis="y", alpha=0.3)

# 2. Distribution PNB 12m
if "pnb_12m_total" in df.columns:
    pnb = df["pnb_12m_total"].dropna()
    axes[0, 1].hist(pnb.clip(upper=pnb.quantile(0.99)), bins=60, color="teal", edgecolor="white")
    axes[0, 1].set_title("Distribution PNB 12 mois")
    axes[0, 1].set_xlabel("PNB (EUR)")
    axes[0, 1].grid(alpha=0.3)

# 3. PNB par segment
if "segment" in df.columns and "pnb_12m_total" in df.columns:
    df.groupby("segment")["pnb_12m_total"].mean().sort_values().plot(
        kind="barh", ax=axes[1, 0], color="coral", edgecolor="white")
    axes[1, 0].set_title("PNB moyen par segment")
    axes[1, 0].set_xlabel("PNB moyen (EUR)")
    axes[1, 0].grid(axis="x", alpha=0.3)

# 4. n_produits
if "n_produits_total" in df.columns:
    df["n_produits_total"].value_counts().sort_index().plot(
        kind="bar", ax=axes[1, 1], color="mediumorchid", edgecolor="white")
    axes[1, 1].set_title("Nombre de produits")
    axes[1, 1].set_xlabel("Produits")
    axes[1, 1].grid(axis="y", alpha=0.3)

plt.suptitle("Donnees Acquisitions — Banque Luxembourg", fontsize=14)
plt.tight_layout()
plt.show()

print("Pret pour les analyses HA1-HA10.")